In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../data/BCH-USDT.csv')
df

,date,open,high,low,close,volume
0,1574935200000,220.08,224.60,212.02,218.98,1329.50718
1,1574938800000,218.98,219.40,218.12,219.04,511.87305
2,1574942400000,219.04,219.05,216.57,217.53,781.62961
3,1574946000000,217.55,218.78,217.54,218.29,461.33172
4,1574949600000,218.00,219.00,217.45,218.67,600.06810
...,...,...,...,...,...,...
31497,1688364000000,297.10,297.10,289.40,292.50,19951.20100
31498,1688367600000,292.50,297.80,291.50,295.30,18255.42300
31499,1688371200000,295.30,296.30,292.30,293.70,12554.16000
31500,1688374800000,293.70,295.00,291.30,291.50,10128.30600


In [3]:
df['date'] = pd.to_datetime(df['date'], unit='ms')
df

,date,open,high,low,close,volume
0,2019-11-28 10:00:00,220.08,224.60,212.02,218.98,1329.50718
1,2019-11-28 11:00:00,218.98,219.40,218.12,219.04,511.87305
2,2019-11-28 12:00:00,219.04,219.05,216.57,217.53,781.62961
3,2019-11-28 13:00:00,217.55,218.78,217.54,218.29,461.33172
4,2019-11-28 14:00:00,218.00,219.00,217.45,218.67,600.06810
...,...,...,...,...,...,...
31497,2023-07-03 06:00:00,297.10,297.10,289.40,292.50,19951.20100
31498,2023-07-03 07:00:00,292.50,297.80,291.50,295.30,18255.42300
31499,2023-07-03 08:00:00,295.30,296.30,292.30,293.70,12554.16000
31500,2023-07-03 09:00:00,293.70,295.00,291.30,291.50,10128.30600


In [4]:
df.set_index(df['date'], inplace=True)
del df['date']
df

,open,high,low,close,volume
date,,,,,
2019-11-28 10:00:00,220.08,224.60,212.02,218.98,1329.50718
2019-11-28 11:00:00,218.98,219.40,218.12,219.04,511.87305
2019-11-28 12:00:00,219.04,219.05,216.57,217.53,781.62961
2019-11-28 13:00:00,217.55,218.78,217.54,218.29,461.33172
2019-11-28 14:00:00,218.00,219.00,217.45,218.67,600.06810
...,...,...,...,...,...
2023-07-03 06:00:00,297.10,297.10,289.40,292.50,19951.20100
2023-07-03 07:00:00,292.50,297.80,291.50,295.30,18255.42300
2023-07-03 08:00:00,295.30,296.30,292.30,293.70,12554.16000


In [5]:
df['ma50'] = df['close'].rolling(50).mean()
df['ma200'] = df['close'].rolling(200).mean()
df

,open,high,low,close,volume,ma50,ma200
date,,,,,,,
2019-11-28 10:00:00,220.08,224.60,212.02,218.98,1329.50718,NaN,NaN
2019-11-28 11:00:00,218.98,219.40,218.12,219.04,511.87305,NaN,NaN
2019-11-28 12:00:00,219.04,219.05,216.57,217.53,781.62961,NaN,NaN
2019-11-28 13:00:00,217.55,218.78,217.54,218.29,461.33172,NaN,NaN
2019-11-28 14:00:00,218.00,219.00,217.45,218.67,600.06810,NaN,NaN
...,...,...,...,...,...,...,...
2023-07-03 06:00:00,297.10,297.10,289.40,292.50,19951.20100,291.716,249.451
2023-07-03 07:00:00,292.50,297.80,291.50,295.30,18255.42300,291.904,249.864
2023-07-03 08:00:00,295.30,296.30,292.30,293.70,12554.16000,291.944,250.265


In [6]:
df['buy_signal'] = False
df['sell_signal'] = False
df.loc[df['ma50'] > df['ma200'], 'buy_signal'] = True
df.loc[df['ma50'] < df['ma200'], 'sell_signal'] = True
df

,open,high,low,close,volume,ma50,ma200,buy_signal,sell_signal
date,,,,,,,,,
2019-11-28 10:00:00,220.08,224.60,212.02,218.98,1329.50718,NaN,NaN,False,False
2019-11-28 11:00:00,218.98,219.40,218.12,219.04,511.87305,NaN,NaN,False,False
2019-11-28 12:00:00,219.04,219.05,216.57,217.53,781.62961,NaN,NaN,False,False
2019-11-28 13:00:00,217.55,218.78,217.54,218.29,461.33172,NaN,NaN,False,False
2019-11-28 14:00:00,218.00,219.00,217.45,218.67,600.06810,NaN,NaN,False,False
...,...,...,...,...,...,...,...,...,...
2023-07-03 06:00:00,297.10,297.10,289.40,292.50,19951.20100,291.716,249.451,True,False
2023-07-03 07:00:00,292.50,297.80,291.50,295.30,18255.42300,291.904,249.864,True,False
2023-07-03 08:00:00,295.30,296.30,292.30,293.70,12554.16000,291.944,250.265,True,False


In [7]:
df['sell_signal'].value_counts()

sell_signal
True     16193
False    15309
Name: count, dtype: int64

In [11]:
balance = 1000
position = None

for index, row in df.iterrows():
    if position is None and row['buy_signal'] is True:
        buy_price = row['close']
        usd_size = balance
        position = {
            'buy_price': buy_price,
            'usd_size': usd_size
        }
        print(f'{index} Bought Bitcoin for {usd_size} of {buy_price}')
    if position and row['sell_signal'] is True:
        sell_price = row['close']
        trade_results = (sell_price - position['buy_price']) / position['buy_price']
        balance += trade_results * position['usd_size']
        print(f'{index} Sold Bitcoin for {balance} of {sell_price} {round(trade_results * 100, 2)} %')
        position = None



2019-12-08 20:00:00 Bought Bitcoin for 1000 of 213.72
2019-12-09 18:00:00 Sold Bitcoin for 976.4177428411006 of 208.68 -2.36 %
2019-12-23 15:00:00 Bought Bitcoin for 976.4177428411006 of 196.61
2019-12-26 05:00:00 Sold Bitcoin for 914.9353581385277 of 184.23 -6.3 %
2019-12-27 19:00:00 Bought Bitcoin for 914.9353581385277 of 201.0
2020-01-03 05:00:00 Sold Bitcoin for 929.5014931934694 of 204.2 1.59 %
2020-01-03 17:00:00 Bought Bitcoin for 929.5014931934694 of 221.32
2020-01-23 17:00:00 Sold Bitcoin for 1353.8080893318042 of 322.35 45.65 %
2020-01-27 14:00:00 Bought Bitcoin for 1353.8080893318042 of 371.6
2020-02-16 19:00:00 Sold Bitcoin for 1445.8714112301213 of 396.87 6.8 %
2020-03-04 00:00:00 Bought Bitcoin for 1445.8714112301213 of 328.94
2020-03-08 22:00:00 Sold Bitcoin for 1211.2370829895185 of 275.56 -16.23 %
2020-03-19 18:00:00 Bought Bitcoin for 1211.2370829895185 of 219.63
2020-03-28 18:00:00 Sold Bitcoin for 1152.117374709012 of 208.91 -4.88 %
2020-04-01 06:00:00 Bought Bitcoi

In [ ]:
print(f'Final balance {round(balance,2)} $')

Final balance 1505.46


In [24]:
class BackTestCrossMA:

    def __init__(self) -> None:
        self.df = pd.DataFrame()

    def load_data(self, path:str) -> None:
        self.df = pd.read_csv(path)
        self.df['date'] = pd.to_datetime(self.df['date'], unit='ms')
        self.df.set_index(self.df['date'], inplace=True)
        del self.df['date']

    def populate_indicators(self):
        self.df['ma100'] = self.df['close'].rolling(100).mean()
        self.df['ma200'] = self.df['close'].rolling(200).mean()

    def populate_signals(self):
        self.df['buy_signal'] = False
        self.df['sell_signal'] = False
        self.df.loc[self.df['ma100'] > self.df['ma200'], 'buy_signal'] = True
        self.df.loc[self.df['ma100'] < self.df['ma200'], 'sell_signal'] = True

    def run_back_test(self):
        balance = 1000
        position = None
        asset = 'BTC'

        for index, row in self.df.iterrows():
            if position is None and row['buy_signal']:
                open_price = row['close']
                usd_size = balance
                position = {
                    'open_price': open_price,
                    'usd_size': usd_size
                }
                print(f'{index} Buy for {balance} of {asset} {usd_size} at {open_price}')
            if position and row['sell_signal']:
                close_price = row['close']
                trade_results = (close_price - position['open_price']) / position['open_price']
                balance += trade_results * position['usd_size']
                position = None
                print(f'{index} Sell for {balance} of {asset} {usd_size} at {open_price}')


        print(f'Final balance {round(balance,2)} $')

bt = BackTestCrossMA()
bt.load_data('../data/BCH-USDT.csv')
bt.populate_indicators()
bt.populate_signals()
bt.run_back_test()
        

2019-12-09 08:00:00 Buy for 1000 of BTC 1000 at 212.11
2019-12-10 15:00:00 Sell for 968.1297440007543 of BTC 1000 at 212.11
2019-12-24 07:00:00 Buy for 968.1297440007543 of BTC 968.1297440007543 at 189.76
2019-12-27 18:00:00 Sell for 1021.1381126778614 of BTC 968.1297440007543 at 189.76
2019-12-27 22:00:00 Buy for 1021.1381126778614 of BTC 1021.1381126778614 at 201.85
2020-01-23 02:00:00 Sell for 1729.2823436050203 of BTC 1021.1381126778614 at 201.85
2020-01-28 14:00:00 Buy for 1729.2823436050203 of BTC 1729.2823436050203 at 374.66
2020-02-17 08:00:00 Sell for 1798.0548491292684 of BTC 1729.2823436050203 at 374.66
2020-03-05 00:00:00 Buy for 1798.0548491292684 of BTC 1798.0548491292684 at 325.35
2020-03-09 13:00:00 Sell for 1492.1617005222142 of BTC 1798.0548491292684 at 325.35
2020-03-20 03:00:00 Buy for 1492.1617005222142 of BTC 1492.1617005222142 at 219.67
2020-03-29 19:00:00 Sell for 1396.45196154849 of BTC 1492.1617005222142 at 219.67
2020-04-02 17:00:00 Buy for 1396.45196154849 o